# Initial Investigations of Alert Cutouts for Comets vs Point-Sources

to **two real** LSST alert packets: 
1. Comet C/2024 E1 (`diaSourceId 170595944383905807`, visit `2026062700881`, detector `36`, band `i`) 
2. Asteroid (`diaSourceId 170631126370484290`, visit `2026070500863`, detector `94`, band `i`)

In this notebook, we load the science/difference/PSF cutouts, run our own PSF and aperture photometry, and compare the two directly.

Self-contained: only `numpy`, `matplotlib`, `astropy`, `scipy`, and `pandas` are required. Expects the alert files in `data/example_alerts/` (relative to the repo root), or update `DATA_DIR` below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
from scipy.optimize import curve_fit

## 1. Shared helper functions

The same closed-form estimators used throughout this project -- PSF photometry is a matched-filter amplitude fit against a *fixed* template (here, the real measured PSF stamp, not an assumed Gaussian), and aperture photometry is a plain circular sum.

In [ ]:
SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))


def gauss(r, amplitude, sigma):
    return amplitude * np.exp(-r**2 / (2 * sigma**2))


def psf_photometry(image, template):
    """Weighted least-squares amplitude fit of `template` to `image` (matched filter)."""
    return np.sum(image * template) / np.sum(template ** 2)


def aperture_photometry(image, center, radius_px):
    """Sum flux within a circular aperture of the given radius (pixels)."""
    yy, xx = np.indices(image.shape)
    rho = np.hypot(xx - center[1], yy - center[0])
    return image[rho <= radius_px].sum()


def embed_psf(psf_stamp, image_shape, center):
    """
    Place psf_stamp (peak at its own array center) into a zero-padded array of
    image_shape, so its center lands at the nearest integer pixel to `center` = (row, col).
    Clips to the overlapping region if the stamp doesn't fully fit inside image_shape
    (the PSF stamp is a fixed 41x41 regardless of the cutout's own size, so a small,
    tightly-cropped cutout can be smaller than the stamp).
    """
    stamp_center_row, stamp_center_col = np.array(psf_stamp.shape) // 2
    row0 = int(round(center[0])) - stamp_center_row
    col0 = int(round(center[1])) - stamp_center_col

    template = np.zeros(image_shape)

    dst_row0, dst_row1 = max(row0, 0), min(row0 + psf_stamp.shape[0], image_shape[0])
    dst_col0, dst_col1 = max(col0, 0), min(col0 + psf_stamp.shape[1], image_shape[1])
    src_row0, src_row1 = dst_row0 - row0, dst_row1 - row0
    src_col0, src_col1 = dst_col0 - col0, dst_col1 - col0

    template[dst_row0:dst_row1, dst_col0:dst_col1] = psf_stamp[src_row0:src_row1, src_col0:src_col1]
    return template


def njy_to_ab_mag(flux_njy):
    """Convert a flux density in nanojansky to an AB apparent magnitude."""
    return -2.5 * np.log10(flux_njy / 3631e9)

## 2. Load and analyze one source

Bundles everything from the single-source version of this notebook into one function: load the FITS cutouts, get the pixel scale from the WCS, embed the real PSF at the source's exact sub-pixel position, and measure PSF + aperture flux ourselves.

In [ ]:
APERTURE_RADII_ARCSEC = [0.6, 1.2, 1.8, 2.4, 3.4, 5.0]


def analyze_source(label, science_path, difference_path):
    """Load one source's FITS cutouts and measure it. Returns a dict of results."""
    science_hdul = fits.open(science_path)
    difference_hdul = fits.open(difference_path)

    science_image = science_hdul['PRIMARY'].data.astype(float)
    difference_image = difference_hdul['PRIMARY'].data.astype(float)
    noise_sigma_map = np.sqrt(difference_hdul['UNCERT'].data.astype(float))
    psf_stamp = science_hdul['PSFIMAGE'].data.astype(float)
    header = science_hdul['PRIMARY'].header

    pixel_scale_arcsec = np.hypot(header['PC1_1'], header['PC2_1']) * 3600
    center = (header['CRPIX2'] - 1, header['CRPIX1'] - 1)  # (row, col), zero-indexed

    psf_template = embed_psf(psf_stamp, difference_image.shape, center)
    our_psf_flux = psf_photometry(difference_image, psf_template)

    our_aperture_flux = {
        r: aperture_photometry(difference_image, center, r / pixel_scale_arcsec)
        for r in APERTURE_RADII_ARCSEC
    }

    # Gaussian fit to the PSF stamp's own radial profile, for a comparable FWHM
    psf_center = np.array(psf_stamp.shape) / 2 - 0.5
    yy, xx = np.indices(psf_stamp.shape)
    psf_r = np.hypot(xx - psf_center[1], yy - psf_center[0])
    pars, _ = curve_fit(gauss, psf_r.ravel(), psf_stamp.ravel(), p0=[psf_stamp.max(), 1.6])
    psf_fwhm_arcsec = pars[1] * SIGMA_TO_FWHM * pixel_scale_arcsec

    return dict(
        label=label,
        science_image=science_image,
        difference_image=difference_image,
        noise_sigma_map=noise_sigma_map,
        psf_stamp=psf_stamp,
        psf_template=psf_template,
        center=center,
        pixel_scale_arcsec=pixel_scale_arcsec,
        our_psf_flux=our_psf_flux,
        our_aperture_flux=our_aperture_flux,
        psf_fwhm_arcsec=psf_fwhm_arcsec,
    )

## 3. Load both alert packets

The two source CSVs use different schemas (the comet's is a Fink-broker export with `r:`-prefixed columns; the asteroid's is a plain `detections` export), so each is read with its own column names into a common, unified dict.

In [ ]:
DATA_DIR = "../data/example_alerts"

COMET_ID = 170595944383905807
ASTEROID_ID = 170631126370484290

In [ ]:
comet_row = pd.read_csv(f"{DATA_DIR}/comet_{COMET_ID}.csv").iloc[0]
comet_alert = dict(
    band=comet_row['r:band'], extendedness=comet_row['r:extendedness'], snr=comet_row['r:snr'],
    bboxSize=comet_row['r:bboxSize'], psfFlux=comet_row['r:psfFlux'], apFlux=comet_row['r:apFlux'],
    ra=comet_row['r:ra'], dec=comet_row['r:dec'], visit=comet_row['r:visit'], detector=comet_row['r:detector'],
)

asteroid_table = pd.read_csv(f"{DATA_DIR}/asteroid_{ASTEROID_ID}_detections.csv")
asteroid_row = asteroid_table[asteroid_table['diaObjectId'] == ASTEROID_ID].iloc[0]
asteroid_alert = dict(
    band=asteroid_row['band_name'], extendedness=asteroid_row['extendedness'], snr=asteroid_row['snr'],
    bboxSize=asteroid_row['bboxSize'], psfFlux=asteroid_row['psfFlux'], apFlux=asteroid_row['apFlux'],
    ra=asteroid_row['ra'], dec=asteroid_row['dec'], visit=asteroid_row['visit'], detector=asteroid_row['detector'],
)

for name, alert in [("Comet", comet_alert), ("Asteroid", asteroid_alert)]:
    print(f"--- {name} ---")
    for k, v in alert.items():
        print(f"  {k:<12} {v}")

## 4. Run our analysis on both sources

In [ ]:
comet = analyze_source(
    "Comet",
    f"{DATA_DIR}/comet_{COMET_ID}_science.fits",
    f"{DATA_DIR}/comet_{COMET_ID}_difference.fits",
)
asteroid = analyze_source(
    "Asteroid",
    f"{DATA_DIR}/asteroid_{ASTEROID_ID}_science.fits",
    f"{DATA_DIR}/asteroid_{ASTEROID_ID}_difference.fits",
)

## 5. Compare: PSF flux, aperture flux, PSF size

Same validation as before (our matched-filter PSF flux vs. the catalog's `psfFlux`; our aperture flux at 2.4" vs. the catalog's `apFlux`), now for both sources side by side.

In [ ]:
for result, alert in [(comet, comet_alert), (asteroid, asteroid_alert)]:
    print(f"--- {result['label']} ---")
    print(f"  band={alert['band']}  extendedness={alert['extendedness']:.3f}  snr={alert['snr']:.1f}")
    print(f"  our psfFlux:  {result['our_psf_flux']:12.2f} nJy   "
          f"catalog: {alert['psfFlux']:12.2f} nJy   "
          f"({100 * (result['our_psf_flux'] / alert['psfFlux'] - 1):+.1f}%)")
    our_ap_2p4 = result['our_aperture_flux'][2.4]
    print(f"  our apFlux (2.4\"): {our_ap_2p4:12.2f} nJy   "
          f"catalog: {alert['apFlux']:12.2f} nJy   "
          f"({100 * (our_ap_2p4 / alert['apFlux'] - 1):+.1f}%)")
    print(f"  PSF FWHM (fit to real stamp): {result['psf_fwhm_arcsec']:.3f}\"")
    print()

## 6. Is the PSF shape actually the same for both sources?

The comet and asteroid are on different visits/detectors/bands, so this is a real test: is the
comet's PSF stamp the same *shape* as the asteroid's, just at a different amplitude -- or are
they two independently-fit models that happen to look similar? Since both `PSFIMAGE` stamps are
already normalized to sum to 1, no rescaling is needed for the comparison; any leftover
difference after that normalization is a real shape difference.

In [ ]:
diff = comet['psf_stamp'] - asteroid['psf_stamp']
identical = np.array_equal(comet['psf_stamp'], asteroid['psf_stamp'])

print(f"Bit-for-bit identical: {identical}")
print(f"Max abs difference:    {np.abs(diff).max():.3e}")
print(f"RMS difference:        {np.sqrt(np.mean(diff**2)):.3e}")
print(f"(RMS of the stamp itself, for scale: {np.sqrt(np.mean(comet['psf_stamp']**2)):.3e})")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (img, title) in zip(axes, [
    (comet['psf_stamp'], "Comet PSF stamp"),
    (asteroid['psf_stamp'], "Asteroid PSF stamp"),
    (diff, "Difference"),
]):
    im = ax.imshow(img, origin="lower", cmap="viridis")
    fig.colorbar(im, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 7. Comet radial profile, binned

Raw pixels (noisy) plus a radially-binned mean, which averages over many pixels at each
radius and cancels out much of the per-pixel noise -- alongside the (unbinned) PSF model
curve for comparison.

In [ ]:
image = comet['difference_image']
center = comet['center']
pixel_scale = comet['pixel_scale_arcsec']
pixel_area = pixel_scale ** 2  # arcsec^2 / pixel, for nJy/pixel -> nJy/arcsec^2

yy, xx = np.indices(image.shape)
rho_arcsec = np.hypot(xx - center[1], yy - center[0]) * pixel_scale

order = np.argsort(rho_arcsec.ravel())
rho_sorted = rho_arcsec.ravel()[order]
values_sorted = image.ravel()[order] / pixel_area
psf_model_sorted = (comet['our_psf_flux'] * comet['psf_template']).ravel()[order] / pixel_area

N_BINS = 20
bin_edges = np.linspace(0, rho_sorted.max(), N_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_indices = np.digitize(rho_sorted, bin_edges) - 1

bin_means = np.full(N_BINS, np.nan)
bin_errs = np.full(N_BINS, np.nan)
for i in range(N_BINS):
    in_bin = values_sorted[bin_indices == i]
    if len(in_bin) > 0:
        bin_means[i] = in_bin.mean()
        bin_errs[i] = in_bin.std() / np.sqrt(len(in_bin))  # standard error of the mean

plt.figure(figsize=(7, 5))
plt.scatter(rho_sorted, values_sorted, s=6, alpha=0.15, color="tab:blue", label="Difference image (pixels)")
plt.errorbar(bin_centers, bin_means, yerr=bin_errs, fmt="o-", color="darkorange", linewidth=2,
             markersize=5, capsize=3, label=f"Radially binned mean ({N_BINS} bins)")
# the model's Gaussian tail underflows toward zero -- drop the physically
# irrelevant deep tail rather than let it drag the log y-axis across hundreds of decades
FLOOR = 1.0 / pixel_area  # nJy/arcsec^2
psf_visible = psf_model_sorted > FLOOR
plt.plot(rho_sorted[psf_visible], psf_model_sorted[psf_visible], color="red", linewidth=2, label="PSF model (scaled)")

# reference 1/rho slope over 2-6", offset above the data so it's visually distinct
rho_ref = np.logspace(np.log10(2), np.log10(6), 50)
rho_mid = np.sqrt(2 * 6)  # geometric midpoint, for anchoring
valid = ~np.isnan(bin_means)
anchor_value = np.interp(rho_mid, bin_centers[valid], bin_means[valid])
OFFSET_FACTOR = 2.5
y_ref = OFFSET_FACTOR * anchor_value * (rho_mid / rho_ref)
plt.plot(rho_ref, y_ref, color="black", linestyle="-", linewidth=2, label="1/ρ reference slope")

plt.xscale("log")
plt.yscale("log")
plt.ylim(bottom=FLOOR)
plt.xlim(left=0.5 * pixel_scale, right=rho_sorted.max() * 1.05)
plt.xlabel("Radius (arcsec)")
plt.ylabel("Surface brightness (nJy/arcsec²)")
plt.title("Comet radial profile (binned)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Combined radial profile

Both sources' difference-image pixels and PSF models, on the same arcsec axis -- watch for
whether each source's binned mean tracks its own PSF model curve (point-like, like the
asteroid) or sits above it at larger radii (extended, like the comet).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for result, psf_color, mean_color in [(comet, "lightblue", "darkblue"), (asteroid, "lightgreen", "darkgreen")]:
    image = result['difference_image']
    center = result['center']
    pixel_scale = result['pixel_scale_arcsec']
    pixel_area = pixel_scale ** 2  # arcsec^2 / pixel, for nJy/pixel -> nJy/arcsec^2

    yy, xx = np.indices(image.shape)
    rho_arcsec = np.hypot(xx - center[1], yy - center[0]) * pixel_scale

    order = np.argsort(rho_arcsec.ravel())
    rho_sorted = rho_arcsec.ravel()[order]
    values_sorted = image.ravel()[order] / pixel_area
    psf_model_sorted = (result['our_psf_flux'] * result['psf_template']).ravel()[order] / pixel_area

    n_bins = 15
    bin_edges = np.linspace(0, rho_sorted.max(), n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_indices = np.digitize(rho_sorted, bin_edges) - 1
    bin_means = np.full(n_bins, np.nan)
    for i in range(n_bins):
        in_bin = values_sorted[bin_indices == i]
        # keep only positive bins for a clean log plot -- noise can dip negative
        if len(in_bin) > 0 and in_bin.mean() > 0:
            bin_means[i] = in_bin.mean()

    # PSF model's Gaussian tail underflows toward zero -- drop the physically
    # irrelevant deep tail rather than let it drag the y-axis across 40 decades
    FLOOR = 1.0 / pixel_area  # nJy/arcsec^2, equivalent to the old 1 nJy/pixel floor
    psf_visible = psf_model_sorted > FLOOR
    ax.plot(rho_sorted[psf_visible], psf_model_sorted[psf_visible], color=psf_color, linewidth=2,
            linestyle="--", label=f"{result['label']}: PSF model")
    ax.plot(bin_centers, bin_means, "o-", color=mean_color, linewidth=2, markersize=5,
            label=f"{result['label']}: binned mean")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(bottom=FLOOR)
ax.set_xlim(left=0.5*pixel_scale)
ax.set_xlabel("Radius (arcsec)")
ax.set_ylabel("Surface brightness (nJy/arcsec²)")
ax.set_title("Radial profile: comet vs. asteroid")
ax.legend()
plt.tight_layout()
plt.show()

## 9. 2D residuals: comet vs. asteroid

Same 2x3 layout as the fake-comet-vs-fake-star comparison earlier in this project: observed / PSF model / residual, one row per source.

In [ ]:
CIRCLE_RADII_ARCSEC = [2.4, 5.0]
CIRCLE_COLORS = ["lime", "magenta"]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle("Comet vs. asteroid: observed, PSF model, residual", fontsize=12)

for row_axes, result in zip(axes, [comet, asteroid]):
    image = result['difference_image']
    model = result['our_psf_flux'] * result['psf_template']
    residual = image - model
    center_xy = result['center'][::-1]  # (row, col) -> (x, y) for plotting
    pixel_scale = result['pixel_scale_arcsec']

    vmax_data = np.percentile(np.abs(image), 99.5)
    vmax_resid = np.percentile(np.abs(residual), 99.5)

    panels = [
        (image, f"{result['label']}: observed", -vmax_data, vmax_data),
        (model, f"{result['label']}: PSF model", -vmax_data, vmax_data),
        (residual, f"{result['label']}: residual", -vmax_resid, vmax_resid),
    ]
    for ax, (img, title, vmin, vmax) in zip(row_axes, panels):
        im = ax.imshow(img, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
        fig.colorbar(im, ax=ax)
        ax.set_title(title)
        ax.set_xlabel("x (pixel)")
        ax.set_ylabel("y (pixel)")

        for radius_arcsec, color in zip(CIRCLE_RADII_ARCSEC, CIRCLE_COLORS):
            circle = plt.Circle(center_xy, radius_arcsec / pixel_scale, edgecolor=color,
                                 facecolor="none", linewidth=1.5, label=f'{radius_arcsec}"')
            ax.add_patch(circle)

axes[0, 0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 10. Magnitudes

AB magnitudes (`njy_to_ab_mag`) from PSF flux and from aperture flux at 2.4" and 5.0",
catalog vs. ours, for both sources.

In [ ]:
for result, alert in [(comet, comet_alert), (asteroid, asteroid_alert)]:
    print(f"--- {result['label']} ---")
    print(f"  psfFlux:        catalog mag={njy_to_ab_mag(alert['psfFlux']):.3f}   "
          f"ours mag={njy_to_ab_mag(result['our_psf_flux']):.3f}")
    print(f"  apFlux (2.4\"):  catalog mag={njy_to_ab_mag(alert['apFlux']):.3f}   "
          f"ours mag={njy_to_ab_mag(result['our_aperture_flux'][2.4]):.3f}")
    print(f"  apFlux (5.0\"):  ours mag={njy_to_ab_mag(result['our_aperture_flux'][5.0]):.3f}   "
          f"(no catalog value at this radius)")
    print()

## 11. Reconstructing bboxSize

Our understanding: bboxSize comes from the detected footprint (pixels above a 5-sigma
significance threshold), dilated outward by roughly `2.4 * psf_sigma`, then clipped to
[30, 102] px.

The key subtlety: real LSST detection doesn't threshold each pixel in isolation. Per
Bosch et al. (2018, eq. 10), the image is first convolved with the PSF (a matched filter)
before thresholding -- this is far more sensitive to diffuse, extended flux like a comet's
coma than judging individual pixels, since it sums flux over the whole PSF footprint at
each position. We compare both approaches below against the comet's own noise map
(the `UNCERT` extension), not an assumed noise level.

In [ ]:
from scipy import ndimage
from scipy.signal import fftconvolve

image = comet['difference_image']
center = comet['center']
noise_sigma_map = comet['noise_sigma_map']
psf_stamp = comet['psf_stamp']  # already normalized to sum ~= 1
pixel_scale = comet['pixel_scale_arcsec']
THRESHOLD_SIGMA = 5.0
MIN_BBOX, MAX_BBOX = 30, 102


def bbox_from_mask(mask, dilation_radius):
    """Bounding box (px) of a footprint mask, padded by dilation_radius, clipped to [MIN_BBOX, MAX_BBOX]."""
    ys, xs = np.nonzero(mask)
    x_extent = xs.max() - xs.min() + 2 * dilation_radius
    y_extent = ys.max() - ys.min() + 2 * dilation_radius
    raw = max(x_extent, y_extent)
    return np.clip(raw, MIN_BBOX, MAX_BBOX), raw, (xs.min(), xs.max(), ys.min(), ys.max())


noise_sigma = np.median(noise_sigma_map)  # a single global value, not per-pixel

# naive: threshold each pixel against a single global noise level
naive_mask = image > THRESHOLD_SIGMA * noise_sigma

# matched-filter: convolve with the PSF first, then threshold the significance map
effective_area = np.sum(psf_stamp ** 2)
convolved = fftconvolve(image, psf_stamp[::-1, ::-1], mode="same")
detection_significance = convolved / (noise_sigma * np.sqrt(effective_area))
matched_mask_all = detection_significance > THRESHOLD_SIGMA

# keep only the connected component containing the source itself -- a real
# footprint is one connected region; separate blobs elsewhere are their own
# (spurious, noise-driven) detections, not part of this source's footprint
labeled, _ = ndimage.label(matched_mask_all)
source_label = labeled[int(round(center[0])), int(round(center[1]))]
matched_mask = labeled == source_label

sigma_psf_px = (comet['psf_fwhm_arcsec'] / SIGMA_TO_FWHM) / pixel_scale
dilation_radius = 2.4 * sigma_psf_px

naive_bbox, naive_raw, naive_extent = bbox_from_mask(naive_mask, dilation_radius)
matched_bbox, matched_raw, matched_extent = bbox_from_mask(matched_mask, dilation_radius)

print(f"naive (per-pixel) footprint:    {naive_mask.sum():5d} px  ->  raw bbox {naive_raw:6.1f}  ->  clipped {naive_bbox:.1f}")
print(f"matched-filter footprint (all):  {matched_mask_all.sum():5d} px")
print(f"matched-filter footprint (connected): {matched_mask.sum():5d} px  ->  raw bbox {matched_raw:6.1f}  ->  clipped {matched_bbox:.1f}")
print(f"catalog bboxSize:                                                              {comet_alert['bboxSize']}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
vmax = np.percentile(image, 99.5)
im = ax.imshow(image, origin="lower", cmap="gray", vmin=-0.2 * vmax, vmax=vmax)
fig.colorbar(im, ax=ax, label="Flux (nJy)")

ax.contour(naive_mask, levels=[0.5], colors="yellow", linewidths=1.5)
ax.contour(matched_mask_all, levels=[0.5], colors="cyan", linewidths=0.8, linestyles=":", alpha=0.6)
ax.contour(matched_mask, levels=[0.5], colors="cyan", linewidths=1.5)

x0, x1, y0, y1 = matched_extent
cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
half = matched_bbox / 2
rect = plt.Rectangle((cx - half, cy - half), 2 * half, 2 * half, edgecolor="red", facecolor="none", linewidth=2)
ax.add_patch(rect)

handles = [
    plt.Line2D([0], [0], color="yellow", label="naive per-pixel footprint"),
    plt.Line2D([0], [0], color="cyan", linestyle=":", alpha=0.6, label="matched-filter, all blobs"),
    plt.Line2D([0], [0], color="cyan", label="matched-filter, connected footprint"),
    plt.Line2D([0], [0], color="red", label="reconstructed bbox (dilated + clipped)"),
]
ax.legend(handles=handles, loc="upper right", fontsize=8)
ax.set_xlabel("x (pixel)")
ax.set_ylabel("y (pixel)")
ax.set_title(f"Comet: catalog bboxSize={comet_alert['bboxSize']}, reconstructed={matched_bbox:.0f}")
plt.tight_layout()
plt.show()

## 12. Open questions

- The asteroid's `extendedness` (~0.05) is now properly low, unlike the previous comparison
  source (~0.99) -- worth checking whether Sections 5-9 actually bear that out (does its
  radial profile/residual look point-like the way its extendedness claims?).
- The asteroid's cutout (30x30) is smaller than the `PSFIMAGE` stamp itself (41x41) --
  `embed_psf` now clips to the overlap, but that means part of the PSF model's wings are
  cut off for this source. Does that bias `our_psf_flux` at all relative to the comet, whose
  cutout comfortably contains the whole stamp?
- How do the two sources' 2.4" aperture-vs-PSF-flux ratios compare? (See Section 5 -- compute
  `apFlux / psfFlux` for each and see which is closer to 1.)
- The comet and asteroid share a band (`i`) but are on different visits and detectors (comet:
  visit `2026062700881`, detector `36`; asteroid: visit `2026070500863`, detector `94`) --
  Section 6 shows a bigger PSF shape difference this time (RMS ~39% of the stamp's own RMS,
  vs. ~11% for the previous pair), consistent with their quite different FWHM (1.41" vs.
  1.09").